In [ ]:
%pip install langchain langchain-community langchain-text-splitters

In [ ]:
%pip install langchain-openai

In [ ]:
%pip install langchain_pinecone

⚠️ Neo4j 지식 그래프 동작 여부 확인 전에도 아래 코드 돌려서 경로 잡아야 함.

In [13]:
import sys
from pathlib import Path

# 현재 노트북 위치 기준으로 프로젝트 루트 잡기
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent  # notebooks의 부모가 프로젝트 루트

sys.path.insert(0, str(ROOT))

print("CWD:", Path.cwd())
print("ROOT:", ROOT)
print("sys.path[0]:", sys.path[0])

CWD: d:\llm\analects_chatbot\notebooks
ROOT: D:\llm\analects_chatbot
sys.path[0]: D:\llm\analects_chatbot


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

with open("./analects_of_confucius.txt", "r", encoding="utf-8") as f:
    text = f.read()

docs = [Document(page_content=text)]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갰을 때 하나의 청크가 가질 수 있는 토큰 수
    chunk_overlap=200,
)

document_list = text_splitter.split_documents(docs)

In [1]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

# Upstage에서 제공하는 Embedding Model을 활용해서 chunk를 vector화
embedding=UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
from langchain_pinecone import PineconeVectorStore
index_name = 'analects-upstage-index'

# Document ID를 지정한 게 아니라면 실행할 때마다 중복된 문서들이 들어갈 수 있다.
# database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

# 한번 저장 후 나중에는 이 코드 활용
database = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

인덱스의 기존 데이터 비우기

In [8]:
from pinecone.grpc import PineconeGRPC as Pinecone
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.environ.get("PINECONE_API_KEY")

if not api_key:
    print("에러: .env 파일에서 API 키를 찾을 수 없습니다.")
else:
    pc = Pinecone(api_key=api_key)
    target_index_name = 'analects-upstage-index'
    
    # 실제 인덱스 객체 생성
    index = pc.Index(target_index_name)

    # 인덱스 내의 모든 벡터 삭제 (인덱스 설정과 주소는 유지됨)
    try:
        index.delete(delete_all=True)
        print(f"'{target_index_name}' 인덱스 초기화 완료!")
    except Exception as e:
        print(f"삭제 중 오류 발생: {e}")




'analects-upstage-index' 인덱스 초기화 완료!


analects_of_confucius.txt의 업데이트를 반영하기

In [15]:
import hashlib
import os
from pinecone.grpc import PineconeGRPC as Pinecone # [수정] 파인콘 클라이언트 불러오기
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_upstage import UpstageEmbeddings
from dotenv import load_dotenv

# 텍스트 파일 읽기 및 청크(Chunk) 분할
def load_and_chunk_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    docs = [Document(page_content=text)]
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
    )
    document_list = text_splitter.split_documents(docs)
    return document_list

# 해시 ID 생성 함수
def generate_id(content):
    return hashlib.md5(content.encode('utf-8')).hexdigest()

# 업로드 실행
def update_index(file_path):
    # 환경변수 로드
    load_dotenv()
    
    # 파인콘 클라이언트 초기화 및 인덱스 연결
    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
    index_name_str = 'analects-upstage-index'
    index = pc.Index(index_name_str) # 실제 인덱스 객체 생성

    chunks = load_and_chunk_file(file_path)
    embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
    
    vectors_to_upsert = []
    
    print(f"총 {len(chunks)}개의 청크를 처리합니다...")

    for chunk in chunks:
        # chunk는 Document 객체이므로 .page_content로 텍스트를 꺼내야 함
        text_content = chunk.page_content 
        
        # 내용이 같으면 ID도 같음 (중복 방지)
        chunk_id = generate_id(text_content)
        
        # 임베딩도 텍스트 내용으로 해야 함
        vector = embeddings.embed_query(text_content)
        
        metadata = {"text": text_content, "source": "analects_file"}
        
        vectors_to_upsert.append({
            "id": chunk_id, 
            "values": vector, 
            "metadata": metadata
        })
    
    # 배치 단위로 Upsert
    batch_size = 100
    for i in range(0, len(vectors_to_upsert), batch_size):
        batch = vectors_to_upsert[i:i+batch_size]
        # index_name(문자열)이 아니라 index(객체)에 upsert 해야 함
        index.upsert(vectors=batch)
        print(f"Upserted batch {i} to {i+len(batch)}")
    
    print("업데이트 완료!")

file_path = "../data/analects_of_confucius.txt"
# 실행
update_index(file_path)

총 84개의 청크를 처리합니다...
Upserted batch 0 to 84
업데이트 완료!


In [3]:
query = '공부 태도를 제대로 하고 싶어.'

# retrieved_docs = database.similarity_search(query, k=3)

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [5]:
from langchain_classic import hub

prompt = hub.pull("rlm/rag-prompt")

In [10]:
from langchain_classic.chains import RetrievalQA

retriever = database.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = retriever,
    chain_type_kwargs = {"prompt":prompt}
)

In [14]:
retriever.invoke(query)

[Document(id='424287ad-c4a8-425b-882b-65b03bd2689d', metadata={}, page_content='공자께서 말씀하셨다. “후배들이란 두려운 것이니, 그들이 지금의 우리만 못하리란 것을 어찌 알 수 있겠는가? 사십, 오십이 되어서도 이름이 알려지지 않는다면, 그 또한 두려워할 만한 사람이 못된다.”\n\n\n\n<子罕第九>23 子曰, “法語之言, 能無從乎? 改之爲貴. 巽與之言, 能無說乎? 繹之爲貴. 說而不繹, 從而不改, 吾末如之何也已矣.”\n\n공자께서 말씀하셨다. “올바른 말로 일러주는 것을 따르지 않을 수 있겠는가? 그러나 중요한 것은 실제로 잘못을 고치는 것이다. 은근하게 타이르는 말에 기뻐하지 않을 수 있겠는가? 그러나 중요한 것은 그 참뜻을 찾아 실천하는 것이다. 기뻐하기만 하고 참뜻을 궁구하지 않거나, 따르기만 하고 실제로 잘못을 고치지 않는다면, 나도 그런 사람은 끝내 어찌 할 수가 없구나.”\n\n\n\n<子罕第九>24 子曰, “主忠信, 毋友不如己者, 過則勿憚改.”\n\n공자께서 말씀하셨다. “성심과 신의를 지키며, 자기만 못한 사람을 벗삼지 말고, 잘못이 있으면 고치기를 주저하지 말아라.”\n\n\n\n<子罕第九>25 子曰, “三軍可奪帥也, 匹夫不可奪志也.”\n\n공자께서 말씀하셨다. “대군의 장수를 빼앗을 수는 있어도, 한 사람의 뜻은 빼앗을 수가 없다.”\n\n\n\n<子罕第九>26 子曰, “衣敝縕袍, 與衣狐貉者立, 而不恥者, 其由也與? ‘不忮不求, 何用不臧?’” 子路終身誦之. 子曰, “是道也, 何足以臧?”\n\n공자께서 말씀하셨다. “해진 솜옷을 입고서 여우나 담비 털가죽옷을 입은 사람과 같이 서 있어도 부끄러워하지 않을 사람이 바로 유로다! 그러나 ‘남을 해치지도 않고 남의 것을 탐내지도 않으니 어찌 훌륭하지 않은가?’라는 시의 한 구절을 자로가 평생 외우고 다니겠다고 하자, 공자께서 말씀하셨다. “그런 도(道)가 어찌 훌륭하다고까지 할 수 있겠느냐?”\n\n\n\n<子罕第九>27 子曰, “歲寒

In [ ]:
ai_message = qa_chain({"query": query})

In [ ]:
ai_message

In [19]:
ai_message2 = qa_chain.invoke({"query": "선생님으로서 학급을 잘 운영하고싶어."})

In [ ]:
ai_message2

## Neo4j 지식 그래프 동작 여부 확인

In [4]:
from kg_pipeline.graph_store import get_neo4j_driver

driver = get_neo4j_driver()
with driver.session() as s:
    print("Total nodes:", s.run("MATCH (n) RETURN count(n) AS c").single()["c"])
    print("Labels:", s.run("CALL db.labels()").value())
driver.close()


[DEBUG][neo4j] URI=neo4j+s://b7d1dbe3.databases.neo4j.io USER=neo4j PWD=SET


Total nodes: 630
Labels: ['Work', 'Chapter', 'Passage', 'Speaker', 'Concept']


In [4]:
from kg_pipeline.graph_store import test_neo4j_connection
print(test_neo4j_connection())


['仁', '君子', '義', '孝', '禮']


In [6]:
import llm
print("USING llm.py:", llm.__file__)

question = "논어에서 효는 개인의 덕목인가, 사회 질서를 위한 기준인가?"
response = llm.get_ai_response(question)

for chunk in response:
    print(chunk, end="")

USING llm.py: D:\llm\analects_chatbot\llm.py


[DEBUG] wrapped_retriever CALLED

[DEBUG] wrapped_retriever CALLED q='논어에서 효는 개인의 덕목인가, 사회 질서를 위한 기준'
[DEBUG] concepts=['仁', '義', '孝', '政', '國', '民', '君', '家']
[DEBUG] strength=strong
[DEBUG] pinecone docs=4
[DEBUG] get_graph_context strength=strong params={'k_paths': 10, 'k_seed_passages': 20, 'max_paths': 5, 'k_passages': 5, 'max_chars': 8000}
[DEBUG] NEO4J_URI env=neo4j+s://b7d1dbe3.databases.neo4j.io
[DEBUG][neo4j] URI=neo4j+s://b7d1dbe3.databases.neo4j.io USER=neo4j PWD=SET
[DEBUG][paths] cypher: // 1) seed concept을 언급하는 passage(p1)들 후보를 먼저 확보
    MATCH (p1:Passage)-[:MENTIONS]->(seed:Concept)
    WHERE seed.name IN $concepts
    WITH p1, collect(DISTINCT seed.name) AS seed_hits, size(collect(DISTINCT seed.name)) AS seed_score
    ORDER BY seed_score DESC
    LIMIT $k_seed_passages

    // 2
[DEBUG][paths] params: {'concepts': ['仁', '義', '孝', '政', '國', '民', '君', '家'], 'k_paths': 10, 'k_seed_passages': 20}
[DEBUG][paths] rows=10 time_ms=1722.1
[DEBUG][paths] first_row_keys: ['relev

《學而第一》2  
「有子 曰 其爲人也 孝弟 而好犯上者 鮮矣 不好犯上 而好作亂者 未之有也 君子 務本 本立而道生 孝弟也者 其爲仁之本與」  
유자가 말하기를, "효성과 공경을 갖춘 사람은 윗사람을 침범하는 것을 좋아하지 않으며, 윗사람을 침범하지 않으면서 질서를 어지럽히는 사람은 없다. 군자는 근본에 힘쓰는 것이니, 근본이 확립되면 따라야 할 올바른 도리가 생겨난다. 효도와 공경은 인(仁)을 실천하는 근본이다."

《子路第十三》20  
「宗族稱孝焉 鄕黨稱弟焉」  
"일가 친척들이 효성스럽다고 칭찬하고, 마을 사람들이 공손하다고 칭찬하는 사람이다."

이 두 구절을 통해 볼 때, 효(孝)는 개인의 덕목인 동시에 사회 질서를 위한 중요한 기준입니다.  
효는 개인의 행동과 마음가짐에서 시작되지만, 그 실천이 사회적 칭찬과 인정으로 이어지며, 사회 전체의 질서를 유지하는 데 기여합니다.  

개인의 효가 가족과 사회에 긍정적 영향을 미치고, 사회 질서의 근간을 이루는 것이 아닐까요?  
효를 실천함으로써, 당신의 주변과 사회에 어떤 변화를 기대할 수 있을까요?